# Fibo Lite — capabilities walkthrough

This notebook demonstrates **Fibo Lite** end to end: install the **`fibo-lite`** package from **AWS CodeArtifact** using the Bria Engine, pull model weights from **Hugging Face**, then run the pipeline on three representative flows—**text to image**, **another image from the same structured prompt**, and **image plus edit instructions**.

**Prerequisites**

- Linux with a **CUDA** GPU (the stack is aimed at **H100 / H200** class hardware).
- **`BRIA_API_TOKEN`** — used to obtain a short-lived CodeArtifact credential for the **`fibo-lite`** repository.
- **`HF_TOKEN`** — for Hugging Face access to the **`briaai/*`** model repos.


## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`fibo-lite`** PyPI repository. The Engine expects your Bria API token in the **`token`** header (set from `BRIA_API_TOKEN`).


In [ ]:
import os

import requests

# BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
# print(f"BRIA_API_TOKEN {BRIA_API_TOKEN}")
# if not BRIA_API_TOKEN:
#     raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

# url = "https://engine.prod.bria-api.com/v2/auth/access/code_artifact"
# resp = requests.get(
#     url,
#     params={"repository": "bria-everywhere-testing"},
#     headers={"api_token": BRIA_API_TOKEN},
#     timeout=60,
# )
# resp.raise_for_status()
# payload = resp.json()

# # Engine response shape: { "result": { "authorization_token", "expiration" }, "request_id" }
# result = payload.get("result")
# if not isinstance(result, dict):
#     raise ValueError("Expected a JSON object with key 'result'.")
# auth_token = result.get("authorization_token")
# if not isinstance(auth_token, str) or not auth_token.strip():
#     raise ValueError("Expected result.authorization_token (non-empty string).")

# CODE_ARTIFACT_PASSWORD = auth_token.strip()
# expiration = result.get("expiration")
# print("CodeArtifact credential acquired." + (f" Expires: {expiration}" if expiration else ""))
CODE_ARTIFACT_PASSWORD = "x

## 2. Install `fibo-lite` from CodeArtifact

Install the package using the **`authorization_token`** from the previous step as the password for the **`bria-fibo-lite`** PyPI simple index (CodeArtifact username **`aws`**).


In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# Pip imports call os.getcwd(). If Jupyter started from a folder that was later removed, inherited cwd breaks pip.
try:
    _pip_cwd = str(Path.cwd())
except FileNotFoundError:
    _pip_cwd = str(Path.home())
    os.chdir(_pip_cwd)

_pip_kw = {"cwd": _pip_cwd}

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--upgrade", "requests", "huggingface_hub"],
    **_pip_kw,
)

FIBO_LITE_INDEX = (
    "https://aws:"
    + quote(CODE_ARTIFACT_PASSWORD, safe="")
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-everywhere-testing/simple/"
)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "fibo-lite",
        "--extra-index-url",
        FIBO_LITE_INDEX,
    ],
    **_pip_kw,
)


## 3. Download model weights from Hugging Face

Download snapshots for the **VLM** and **FIBO Lite** checkpoints using `hf_token`, then pass the **local directories** into the pipeline configuration (`VLMConfig.model_path` and `FIBOInternalConfig.model_path`).

Repos follow the defaults used in Bria pipelines (`briaai/FIBO-vlm`, `briaai/FIBO-lite` with a fallback name where applicable).


In [ ]:
import os
from pathlib import Path

from huggingface_hub import snapshot_download

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN for Hugging Face (gated briaai models).")

hf_cache = Path(os.environ.get("HF_HOME", "~/.cache/huggingface")).expanduser()

def _download_fibo_lite_weights(token: str, cache_dir: Path) -> tuple[str, str]:
    vlm_path = snapshot_download(repo_id="briaai/FIBO-vlm", token=token, cache_dir=str(cache_dir))
    fibo_path = snapshot_download(repo_id="briaai/FIBO-lite", token=token, cache_dir=str(cache_dir))
    return vlm_path, fibo_path

vlm_model_path, fibo_lite_model_path = _download_fibo_lite_weights(hf_token, hf_cache)
print("VLM weights:", vlm_model_path)
print("FIBO-lite weights:", fibo_lite_model_path)


## 4. Build configuration and start the pipeline

`compile_model=True` compiles the diffuser for lower latency after warmup; **`compile_model=False`** skips that path and is often easier for first runs. Tune both flags for your GPU and workload.

TorchInductor cache environment variables below help avoid repeated compilation cost across runs.


In [ ]:
import logging
import os
import time

import torch
from IPython.display import display

from fibo_inference.config import FIBOInternalConfig
from fibo_lite.config import FiboLiteConfig, FiboLiteDiffuserConfig
from fibo_lite.fibo_lite import FiboLite
from fibo_lite.schemas import FiboLiteInput
from fibo_lite.vlm.config import VLMConfig

logging.basicConfig(level=logging.INFO)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Fibo Lite.")

fibo_lite_config = FiboLiteConfig(
    vlm_config=VLMConfig(
        model_path=vlm_model_path,
        max_model_len=1024,
        gpu_memory_utilization=0.4,
    ),
    fibo_lite_config=FiboLiteDiffuserConfig(
        compile_model=False,
        inference_config=FIBOInternalConfig(model_path=fibo_lite_model_path),
    ),
    model_path=fibo_lite_model_path,
)

pipeline = FiboLite(config=fibo_lite_config)
pipeline.setup()
print("Pipeline setup complete.")


## 5. Showcase — three Fibo Lite flows

1. **Generate** — plain-language prompt: the VLM turns it into structured JSON, then the diffuser renders an image.
2. **Regenerate from structured prompt** — take the **`structured_prompt`** from step 1 and run the diffuser again (no second VLM call).
3. **Refine from image** — supply an image URL plus edit text: the VLM updates the structured prompt, then the diffuser generates.


In [ ]:
# 5a — Text-to-image (VLM structured prompt + generation)
task_generate = FiboLiteInput(
    prompt=(
        "An american eagle flying over the flag of the United States. "
        "You can hear freedom ringing."
    ),
)
t0 = time.time()
out_gen = pipeline.execute(task_generate)
print(f"generate: {time.time() - t0:.1f}s")
display(out_gen.image)


In [ ]:
# 5b — Same structured prompt as step 5a (from VLM output), new image (no VLM call)
task_struct = FiboLiteInput(structured_prompt=out_gen.structured_prompt)
t0 = time.time()
out_struct = pipeline.execute(task_struct)
print(f"regenerate_from_structured: {time.time() - t0:.1f}s")
display(out_struct.image)


In [ ]:
# 5c — Image + editing instructions (VLM refine + generation)
task_img = FiboLiteInput(
    prompt="add a car to the image",
    image="https://bria-test-images.s3.us-east-1.amazonaws.com/highway.jpg",
)
t0 = time.time()
out_img = pipeline.execute(task_img)
print(f"refine_image: {time.time() - t0:.1f}s")
display(out_img.image)


## 6. Cleanup

Release GPU memory held by the pipeline.


In [ ]:
pipeline.cleanup()
print("Cleanup done.")
